# After-launch spend model (team version)

This notebook estimates **current total_spend** after a campaign has started, using activity already measured for that campaign. It does **not** predict spend before launch or the final spend of an ongoing campaign.

Run the cells from top to bottom. Add both CSV files to Kaggle Input first. If you run locally, put them beside the notebook or in your Downloads folder.

## 1. Load the two datasets

We use the original CSV for the model inputs and target. The cleaned CSV supplies consistent company names so related campaign rows stay together during testing.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, cross_val_score
from sklearn.pipeline import make_pipeline

def find_csv(name):
    places = [Path(name), Path.home() / "Downloads" / name]
    kaggle = Path("/kaggle/input")
    if kaggle.exists():
        places += list(kaggle.rglob(name))
    file = next((place for place in places if place.is_file()), None)
    if file is None:
        raise FileNotFoundError(f"Add {name} to Kaggle Input or put it beside this notebook.")
    return file

original_path = find_csv("BreakThroughTech_training_data_1_3.csv")
cleaned_path = find_csv("combined_dataset.csv")

original = pd.read_csv(original_path)
cleaned = pd.read_csv(cleaned_path)
print("Original:", original_path, original.shape)
print("Cleaned:", cleaned_path, cleaned.shape)

## 2. Keep only real target values

Three rows have a missing original total_spend value. The cleaned CSV estimated those values, but we do not train or test on estimates.

The checks below make sure the two files still have matching rows. If a check fails, stop and compare the CSV versions.

In [ ]:
assert len(original) == len(cleaned), "The two files have different row counts."
assert original["brand"].equals(cleaned["brand"]), "The rows are not in the same order."

real_target = original["total_spend"].notna()
assert real_target.equals(cleaned["total_campaign_budget_source"].eq("Original"))
assert np.allclose(
    original.loc[real_target, "total_spend"],
    cleaned.loc[real_target, "total_campaign_budget"]
), "The target values do not match."

brand_columns = [c for c in cleaned.columns if c.startswith("brand_clean_")]
brand_flags = cleaned.loc[real_target, brand_columns]
assert brand_flags.eq(1).sum(axis=1).eq(1).all(), "A row has no clear cleaned brand."

data = original.loc[real_target].copy()
data["brand_clean"] = brand_flags.idxmax(axis=1).str.replace(
    "brand_clean_", "", regex=False
)
data = data.reset_index(drop=True)

print("Rows used:", len(data))
print("Rows excluded:", len(original) - len(data))

## 3. Pick the inputs and target

These five columns measure how many healthcare professionals the campaign reached and engaged. They are available only after launch. We leave out columns such as actual_spend_per_engagement and log_totalSpend because they are calculated from the answer we are trying to predict.

The random forest can use the raw numeric columns. We do not need to normalize or log-transform them.

In [ ]:
feature_cols = [
    "unique_reach",
    "unique_engaged_hcps",
    "total_engagements",
    "engagement_rate",
    "engagement_frequency",
]

X = data[feature_cols].apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(data["total_spend"], errors="coerce")

dates = pd.to_datetime(data["campaign_start_date"], errors="coerce", utc=True)
assert dates.notna().all(), "A row with a real target has no start date."

# One group = one cleaned company and one exact campaign start time.
groups = data["brand_clean"] + "|" + dates.astype(str)

print("Inputs:", feature_cols)
print("Missing values in inputs:")
print(X.isna().sum())
print("Campaign groups:", groups.nunique())

## 4. Separate campaigns for training and testing

Rows from the same company and start time may describe related tactics. We keep those rows together, so the model cannot see part of a campaign during training and another part during testing.

In [ ]:
split = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_rows, test_rows = next(split.split(X, y, groups))

X_train, X_test = X.iloc[train_rows], X.iloc[test_rows]
y_train, y_test = y.iloc[train_rows], y.iloc[test_rows]
train_groups = groups.iloc[train_rows]

print("Training rows:", len(train_rows))
print("Test rows:", len(test_rows))
print("Groups shared between train and test:",
      len(set(train_groups) & set(groups.iloc[test_rows])))

## 5. Train the random forest

The imputer fills any missing inputs using the median from the training data. It is inside the pipeline so each cross-validation fold learns its own median. The forest settings match the model we tested earlier.

In [ ]:
model = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestRegressor(
        n_estimators=150,
        max_depth=8,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=1,
    ),
)

cv = GroupKFold(n_splits=5)
cv_r2 = cross_val_score(
    model, X_train, y_train,
    groups=train_groups,
    cv=cv,
    scoring="r2",
)

print("R² for each training fold:", np.round(cv_r2, 3))
print("Mean cross-validation R²:", round(cv_r2.mean(), 3))

## 5a. Why these five inputs?

This optional check compares three simple choices using only the training folds. The last option includes tactic spend already measured after launch. We keep the five activity columns because they have the highest average cross-validation R² here. Do not choose inputs by looking at the separate test score.

In [ ]:
options = {
    "3 counts": ["unique_reach", "unique_engaged_hcps", "total_engagements"],
    "5 activity inputs": feature_cols,
    "5 activity + tactic spend": feature_cols + ["total_budget_spent"],
}

comparison = []
for name, columns in options.items():
    X_option = data[columns].apply(pd.to_numeric, errors="coerce")
    scores = cross_val_score(
        model, X_option.iloc[train_rows], y_train,
        groups=train_groups, cv=cv, scoring="r2"
    )
    comparison.append({"Inputs": name, "Mean training CV R²": scores.mean()})

print(pd.DataFrame(comparison).round(3).to_string(index=False))

## 6. Check the separate test campaigns

R² is better when it is higher. An R² near 0 means the model is little better than predicting the training average. MAE is the average size of an error, in the same source units as total_spend.

We use this test set after choosing the model, not to tune its settings.

In [ ]:
model.fit(X_train, y_train)
test_predictions = model.predict(X_test)

mean_prediction = np.repeat(y_train.mean(), len(y_test))

print("Random forest test R²:", round(r2_score(y_test, test_predictions), 3))
print("Random forest test MAE:", f"{mean_absolute_error(y_test, test_predictions):,.0f}")
print("Training-mean baseline R²:", round(r2_score(y_test, mean_prediction), 3))

# With the supplied CSVs, expect about 0.67 for the random forest test R².

## 7. Look at the predictions

Points on the dashed line would be perfect predictions. The plot helps us see which campaigns the model gets wrong, especially very large ones.

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(y_test / 1e9, test_predictions / 1e9, alpha=0.7)
limit = max(y_test.max(), test_predictions.max()) / 1e9
plt.plot([0, limit], [0, limit], "--", color="gray")
plt.xlabel("Actual total_spend (billions of source units)")
plt.ylabel("Predicted total_spend (billions of source units)")
plt.title("Held-out campaign predictions")
plt.tight_layout()
plt.show()

## 8. Check the newest campaigns

The random test above mixes older and newer campaigns. This second check trains on older campaign groups and tests on the newest 20% of groups. It asks a harder question: does the model still work for later campaigns? Do not use this second test to keep changing the model until its score looks good.

In [ ]:
group_dates = pd.DataFrame({
    "group": groups,
    "date": dates,
}).groupby("group")["date"].min().sort_values()

n_newest = int(np.ceil(0.20 * len(group_dates)))
newest_groups = set(group_dates.tail(n_newest).index)
is_newest = groups.isin(newest_groups)

time_model = clone(model)
time_model.fit(X.loc[~is_newest], y.loc[~is_newest])
newest_predictions = time_model.predict(X.loc[is_newest])

print("Older training rows:", (~is_newest).sum())
print("Newer test rows:", is_newest.sum())
print("Newest-campaign R²:",
      round(r2_score(y.loc[is_newest], newest_predictions), 3))
print("Newest-campaign MAE:",
      f"{mean_absolute_error(y.loc[is_newest], newest_predictions):,.0f}")

# With the supplied CSVs, expect a negative newest-campaign R².

## What to tell the team

The five-input model gets about 0.67 R² on a random campaign-group test, but it scores below zero on the newest campaigns. That means the 0.67 result should not be called a reliable forecast for future campaigns. This notebook estimates current spend from activity measured at the same snapshot.

If the team tests another input, first ask: **Would we know this value at the moment we make the prediction?** Do not add a column calculated from total_spend. Keep the campaign-group split and the newest-campaign check when comparing models.